In [ ]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys

chemical_data = pd.read_csv('/workspaces/toxicogenomics_paper/HTTr_data/input/metadata/SMILES_data.csv')  # Assume this CSV has a 'SMILES' column


def maccs_bits(smiles, failed_smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Warning: could not parse SMILES '{smiles}'")
        failed_smiles.append(smiles)
        return [None]*167
    bitvec = MACCSkeys.GenMACCSKeys(mol)
    return list(map(int, bitvec.ToBitString()))

failed_smiles = []
maccs_df = chemical_data['SMILES'] \
              .apply(lambda s: maccs_bits(s, failed_smiles) if pd.notna(s) else [None]*167) \
              .apply(pd.Series)

maccs_df.columns = [f"MACCS_{i}" for i in range(167)]

full_chemical_data = pd.concat([chemical_data, maccs_df], axis=1)

In [ ]:
assay_data    = pd.read_csv("/home/gcattebeke/projects/20250612_machine_learning/papermill/pipeline/all_assays_merged.csv")
httr_data     = pd.read_feather("../HTTr_differential_expression/activity_mapping/signature_scores_all_databases.feather")

chemical_httr_merged = full_chemical_data.merge(httr_data, left_on='dsstox_substance_id', right_on='outcome_id', how='left')
chemical_httr_merged = chemical_httr_merged.dropna(subset=[col for col in chemical_httr_merged.columns if col.startswith("MACCS_")])
chemical_httr_merged = chemical_httr_merged.loc[:, chemical_httr_merged.nunique() > 1]
chemical_httr_merged = chemical_httr_merged.drop(columns=['casn', 'dsstox_substance_id', 'SMILES', 'MS_READY_SMILES', 'QSAR_READY_SMILES', 'MS_READY_MASS', 'MS_READY_FORMULAE'])

chemical_httr_assay_merged = chemical_httr_merged.merge(assay_data, left_on='outcome_id', right_on='dtxsid', how='left').drop(columns=['dtxsid'])
chemical_httr_assay_merged.head()